# Tugas Praktikum - Wisconsin Breast Cancer

## Langkah 0 - Import Library
Mengimpor library yang diperlukan untuk manipulasi dataset dan pra-pengolahan data.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

## Langkah 1 - Memuat Dataset
Memuat file `wbc.csv` ke dalam DataFrame Pandas.

In [3]:
data_path = r'..\Dataset\wbc.csv'
df = pd.read_csv(data_path)
print(f'Dimensi awal dataset: {df.shape}')
df.head()

Dimensi awal dataset: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## Inspeksi Data Awal
Mengecek ringkasan tipe data dan keberadaan missing value pada setiap kolom.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [5]:
# Pengecekan missing value per kolom
null_counts = df.isnull().sum()
print('Kolom dengan nilai kosong (null):')
print(null_counts[null_counts > 0])

Kolom dengan nilai kosong (null):
Unnamed: 32    569
dtype: int64


## Soal 1: Pemisahan Variabel yang Dapat Digunakan dan Tidak Digunakan

**Analisis Variabel:**
- Kolom `id`: Merupakan identitas unik pasien, tidak memiliki nilai prediktif terhadap sifat biologis kanker, sehingga tidak digunakan.
- Kolom `Unnamed: 32`: Merupakan kolom kosong hasil parsing delimiter akhir CSV (100% missing values), sehingga harus dibuang.
- Kolom `diagnosis`: Merupakan variabel target (label klasifikasi).
- Kolom fitur numerik (30 kolom fitur pengukuran sel): Digunakan sebagai variabel prediktor (fitur).

In [6]:
# Menghapus kolom 'id' dan 'Unnamed: 32'
cols_to_drop = [col for col in ['id', 'Unnamed: 32'] if col in df.columns]
df_clean = df.drop(columns=cols_to_drop)

print(f'Dimensi dataset setelah menghapus kolom yang tidak terpakai: {df_clean.shape}')
df_clean.head()

Dimensi dataset setelah menghapus kolom yang tidak terpakai: (569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Soal 2: Encoding pada Kolom 'diagnosis'

Kolom `diagnosis` berisi data kategori kualitatif yaitu `'M'` (Malignant / Ganas) dan `'B'` (Benign / Jinak).
Kita melakukan **Label Encoding** sehingga:
- `B` (Benign) -> `0`
- `M` (Malignant) -> `1`

In [7]:
# Pengecekan distribusi kelas sebelum encoding
print('Distribusi kelas sebelum encoding:')
print(df_clean['diagnosis'].value_counts())

# Proses Label Encoding
le = LabelEncoder()
df_clean['diagnosis'] = le.fit_transform(df_clean['diagnosis'])

print('\nDistribusi kelas setelah encoding (0 = Benign, 1 = Malignant):')
print(df_clean['diagnosis'].value_counts())
df_clean[['diagnosis']].head()

Distribusi kelas sebelum encoding:
diagnosis
B    357
M    212
Name: count, dtype: int64

Distribusi kelas setelah encoding (0 = Benign, 1 = Malignant):
diagnosis
0    357
1    212
Name: count, dtype: int64


,diagnosis
0,1
1,1
2,1
3,1
4,1


## Soal 3: Standardisasi pada Semua Kolom Numerik

Seluruh kolom fitur selain `diagnosis` bertipe numerik kontinu (`float64`) dengan rentang skala yang sangat bervariasi (misalnya `area_mean` berada di kisaran ratusan/ribuan, sedangkan `smoothness_mean` di bawah 0.2).
Oleh karena itu, dilakukan **standardisasi** menggunakan `StandardScaler` sehingga setiap fitur memiliki rata-rata (mean) = 0 dan variansi = 1.

In [8]:
# Mengidentifikasi semua kolom numerik kecuali target 'diagnosis'
numeric_features = [col for col in df_clean.columns if col != 'diagnosis']
print(f'Jumlah fitur numerik yang distandarisasi: {len(numeric_features)}')

# Standardisasi dengan StandardScaler
scaler = StandardScaler()
df_clean[numeric_features] = scaler.fit_transform(df_clean[numeric_features])

# Menampilkan sampel hasil setelah standardisasi
df_clean.head()

Jumlah fitur numerik yang distandarisasi: 30


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,1.097064,-2.073335,1.269934,0.984375,1.568466,3.283515,2.652874,2.532475,2.217515,...,1.886690,-1.359293,2.303601,2.001237,1.307686,2.616665,2.109526,2.296076,2.750622,1.937015
1,1,1.829821,-0.353632,1.685955,1.908708,-0.826962,-0.487072,-0.023846,0.548144,0.001392,...,1.805927,-0.369203,1.535126,1.890489,-0.375612,-0.430444,-0.146749,1.087084,-0.243890,0.281190
2,1,1.579888,0.456187,1.566503,1.558884,0.942210,1.052926,1.363478,2.037231,0.939685,...,1.511870,-0.023974,1.347475,1.456285,0.527407,1.082932,0.854974,1.955000,1.152255,0.201391
3,1,-0.768909,0.253732,-0.592687,-0.764464,3.283553,3.402909,1.915897,1.451707,2.867383,...,-0.281464,0.133984,-0.249939,-0.550021,3.394275,3.893397,1.989588,2.175786,6.046041,4.935010
4,1,1.750297,-1.151816,1.776573,1.826229,0.280372,0.539340,1.371011,1.428493,-0.009560,...,1.298575,-1.466770,1.338539,1.220724,0.220556,-0.313395,0.613179,0.729259,-0.868353,-0.397100


## Verifikasi Hasil Standardisasi
Memverifikasi bahwa mean setiap fitur numerik kini mendekati 0 dan standar deviasi mendekati 1.

In [9]:
# Verifikasi mean dan standar deviasi
summary = pd.DataFrame({
    'Mean': df_clean[numeric_features].mean(),
    'Std': df_clean[numeric_features].std()
})
print('Sampel 5 fitur setelah standardisasi:')
summary.head()

Sampel 5 fitur setelah standardisasi:


,Mean,Std
radius_mean,-1.373633e-16,1.00088
texture_mean,6.868164e-17,1.00088
perimeter_mean,-1.248757e-16,1.00088
area_mean,-2.185325e-16,1.00088
smoothness_mean,-8.366672e-16,1.00088


## Kesimpulan Tugas Praktikum JS02
1. Kolom tidak relevan (`id`) dan kolom kosong (`Unnamed: 32`) berhasil dipisahkan dan dieliminasi dari dataset.
2. Kolom target `diagnosis` berhasil di-encode secara biner menjadi `0` (Benign) dan `1` (Malignant).
3. Seluruh 30 fitur numerik telah berhasil distandarisasi menggunakan z-score scaling, sehingga data siap digunakan pada algoritma pembelajaran mesin.